In [1]:

import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D


In [6]:
# Load Dataset
df = pd.read_csv("spam.csv", encoding="latin-1")

In [7]:

df = df[['v1', 'v2']]
df.columns = ['label', 'message']

KeyError: "None of [Index(['v1', 'v2'], dtype='object')] are in the [columns]"

In [ ]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

In [ ]:
X = df['message']
y = df['label']

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
X_train = tf.convert_to_tensor(X_train.tolist(), dtype=tf.string)
X_test  = tf.convert_to_tensor(X_test.tolist(), dtype=tf.string)

# Convert labels to TensorFlow integer tensors
y_train = tf.convert_to_tensor(y_train.values, dtype=tf.int32)
y_test  = tf.convert_to_tensor(y_test.values, dtype=tf.int32)

In [ ]:
VOCAB_SIZE = 10000
SEQUENCE_LENGTH = 100

vectorizer = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=SEQUENCE_LENGTH
)

In [ ]:
vectorizer.adapt(X_train)

In [ ]:

model = Sequential([
    vectorizer,
    Embedding(VOCAB_SIZE, 16),
    GlobalAveragePooling1D(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [ ]:

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    validation_data=(X_test, y_test)
)

Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8341 - loss: 0.4402 - val_accuracy: 0.8655 - val_loss: 0.3694
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8644 - loss: 0.3682 - val_accuracy: 0.8655 - val_loss: 0.3632
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8665 - loss: 0.3538 - val_accuracy: 0.8655 - val_loss: 0.3450
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8597 - loss: 0.3370 - val_accuracy: 0.8655 - val_loss: 0.3037
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8711 - loss: 0.2729 - val_accuracy: 0.8780 - val_loss: 0.2404
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9060 - loss: 0.2057 - val_accuracy: 0.9363 - val_loss: 0.1618
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9590 - loss: 0.1232 - val_accuracy: 0.9713 - val_loss: 0.1195
Epoch 8/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9753 - loss: 0.0874 - val_accuracy: 0.

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9804 - loss: 0.0713
Test Accuracy: 0.9766815900802612


In [ ]:
history_dict = history.history

In [ ]:
# Make Predictions
sample_sms = tf.constant([
    "Congratulations! You have won a free lottery ticket",
    "Hey, are we still meeting today?"
])

predictions = model.predict(sample_sms)

for sms, pred in zip(sample_sms.numpy(), predictions):
    print("Message:", sms.decode())
    print("Prediction:", "Spam" if pred > 0.5 else "Ham")
    print("-" * 50)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Message: Congratulations! You have won a free lottery ticket
Prediction: Ham
--------------------------------------------------
Message: Hey, are we still meeting today?
Prediction: Ham
--------------------------------------------------
